# 데이터 개요

DACON 온라인 채널 제품 판매 데이터셋을 살펴보고, wide 형식의 판매 매트릭스를 long 시계열 테이블로 변환합니다.


## 데이터 로드

In [ ]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path("data")
train_path = DATA_DIR / "train.csv"
sample_submission_path = DATA_DIR / "sample_submission.csv"

In [ ]:
train = pd.read_csv(train_path)
sample_submission = pd.read_csv(sample_submission_path)

print(f"train: {train.shape}")
print(f"sample_submission: {sample_submission.shape}")
train.head()

## 컬럼 구조 파악

날짜 컬럼과 메타데이터 컬럼을 분리하고 판매 데이터의 기간을 확인합니다.


In [ ]:
parsed_columns = pd.to_datetime(pd.Index(train.columns), errors="coerce")
date_columns = [column for column, parsed in zip(train.columns, parsed_columns) if not pd.isna(parsed)]
metadata_columns = [column for column in train.columns if column not in date_columns]

print(f"메타데이터 컬럼: {metadata_columns}")
print(f"판매 데이터 기간: {date_columns[0]} ~ {date_columns[-1]}")
print(f"판매일수: {len(date_columns)}")

## Wide → Long 변환

제품 × 날짜 형태의 wide 테이블을 long 형식으로 변환합니다.


In [ ]:
sales_long = train.melt(
    id_vars=metadata_columns,
    value_vars=date_columns,
    var_name="date",
    value_name="sales",
)
sales_long["date"] = pd.to_datetime(sales_long["date"])

sales_long.head()

## 일별 전체 판매량 분포

In [ ]:
daily_sales = sales_long.groupby("date", as_index=False)["sales"].sum()
daily_sales.describe()